# extra_trees_rank 原生 ranking adapter 验证

目的：把 QuantBench 模型矩阵中最后一个 `proxy_feature_adapter`（`extra_trees_rank`）升级为稳定的 `native_library` 实现，并验证训练、预测、manifest fidelity 和 smoke 回测输出。

外部参考：`sklearn.ensemble.ExtraTreesRegressor` 是随机极端树回归器；本仓库在其外层增加 `ExtraTreesRankRegressor`，统一接收 forward return 的 rank-normalized target，并显式记录 ranking objective 元数据。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise RuntimeError("Cannot locate repository root containing src/")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model import QUANTBENCH_MODEL_NAMES
from src.model.capabilities import model_capability_table, model_fidelity_summary
from src.model.matrix import run_quantbench_matrix_smoke
from src.model.training import (
    ExtraTreesRankRegressor,
    fit_estimator_for_model,
    fit_predict_model,
    model_adapter_name,
    model_adapter_status,
)

EXPERIMENT_DIR = ROOT / "research/extra_trees_rank_native_adapter_v1"
CARD_DIR = EXPERIMENT_DIR / "matrix_card"
CARD_DIR.mkdir(parents=True, exist_ok=True)

frame = load_soxl_support_frame(start="2026-01-01", end="2026-06-09")
dataset = build_supervised_dataset(frame, horizon=1)
train = dataset.frame.loc[dataset.frame["date"] <= "2026-04-30"].copy()
test = dataset.frame.loc[dataset.frame["date"] > "2026-04-30"].copy()
feature_columns = dataset.feature_columns[:6]

len(QUANTBENCH_MODEL_NAMES), train.shape, test.shape, feature_columns

(32,
 (82, 18),
 (26, 18),
 ('soxl_return_1d',
  'soxx_return_1d',
  'soxl_return_5d',
  'soxx_return_5d',
  'soxl_soxx_relative_5d',
  'vix_level'))

In [2]:
estimator, constant_prediction = fit_estimator_for_model(
    "extra_trees_rank",
    train=train,
    feature_columns=feature_columns,
)
predictions = fit_predict_model(
    "extra_trees_rank",
    train=train,
    test=test,
    feature_columns=feature_columns,
)

assert constant_prediction is None
assert isinstance(estimator, ExtraTreesRankRegressor)
assert estimator.objective_name_ == "ranking"
assert estimator.target_kind_ == "forward_return_rank"
assert abs(estimator.rank_target_mean_) < 1e-12
assert estimator.rank_target_std_ > 0.0
assert estimator.feature_count_ == len(feature_columns)
assert model_adapter_status("extra_trees_rank") == "paper_strict_adapter"
assert model_adapter_name("extra_trees_rank") == "extra_trees_rank_forest_rank_v1"
assert predictions["prediction"].notna().all()
assert predictions["objective"].unique().tolist() == ["ranking"]
assert predictions["model_adapter"].unique().tolist() == ["extra_trees_rank_forest_rank_v1"]

predictions.head()

,date,model,prediction,target_return,objective,family,adapter_status,model_adapter
0,2026-05-01,extra_trees_rank,-0.104227,-0.021856,ranking,tree,paper_strict_adapter,extra_trees_rank_forest_rank_v1
1,2026-05-04,extra_trees_rank,0.065543,0.130223,ranking,tree,paper_strict_adapter,extra_trees_rank_forest_rank_v1
2,2026-05-05,extra_trees_rank,-0.154681,0.150458,ranking,tree,paper_strict_adapter,extra_trees_rank_forest_rank_v1
3,2026-05-06,extra_trees_rank,0.538256,-0.082906,ranking,tree,paper_strict_adapter,extra_trees_rank_forest_rank_v1
4,2026-05-07,extra_trees_rank,0.085906,0.163314,ranking,tree,paper_strict_adapter,extra_trees_rank_forest_rank_v1


In [3]:
capability = model_capability_table()
fidelity = model_fidelity_summary()
extra_trees_row = capability.loc[capability["model"].eq("extra_trees_rank")].iloc[0].to_dict()

assert extra_trees_row["paper_fidelity"] == "native_library"
assert extra_trees_row["is_proxy"] is False
assert fidelity["model_count"] == len(QUANTBENCH_MODEL_NAMES)
assert fidelity["fidelity_counts"].get("proxy_feature_adapter", 0) == 0
assert fidelity["fidelity_counts"]["native_library"] == 10
assert fidelity["proxy_models"] == []
assert "extra_trees_rank" in fidelity["native_models"]

extra_trees_row, fidelity

({'model': 'extra_trees_rank',
  'family': 'tree',
  'objective': 'ranking',
  'input_kind': 'tabular',
  'estimator': 'ExtraTreesRankRegressor',
  'paper_fidelity': 'native_library',
  'is_proxy': False,
  'implementation_notes': 'Uses the native ExtraTreesRankRegressor training path behind the QuantBench-style registry contract.',
  'promotion_blocker': '',
  'trade_instrument': 'SOXL',
  'support_universe': 'SOXL,SOXX,VIX,QQQ,SPY'},
 {'schema_version': 'model.fidelity.v1',
  'model_count': 32,
  'fidelity_counts': {'native_graph': 9,
   'native_library': 10,
   'native_sequence': 7,
   'native_torch': 6},
  'proxy_models': [],
  'native_models': ['alstm_mse',
   'autoformer_mse',
   'catboost_rank',
   'darnn_mse',
   'dlinear_mse',
   'extra_trees_rank',
   'fedformer_mse',
   'gat_mse',
   'gcn_mse',
   'gru_mse',
   'hawkes_gru_mse',
   'hist_gradient_mse',
   'hypergraph_mse',
   'lightgbm_mse',
   'logistic_clf',
   'lstm_mse',
   'market_graph_mse',
   'mlp_mixer_mse',
   'mlp

In [4]:
matrix_outputs = run_quantbench_matrix_smoke(
    card_dir=CARD_DIR,
    start="2026-01-01",
    end="2026-06-09",
    train_end="2026-04-30",
    model_names=(
        "extra_trees_rank",
        "ridge_ic",
        "ridge_mse",
        "hist_gradient_mse",
        "xgboost_mse",
    ),
)
matrix_summary = matrix_outputs.summary

assert matrix_summary["model_fidelity"]["model_count"] == 5
assert matrix_summary["model_fidelity"]["fidelity_counts"].get("proxy_feature_adapter", 0) == 0
assert "extra_trees_rank" in matrix_summary["model_fidelity"]["native_models"]

matrix_summary

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'adapter_status': 'paper_strict_adapter',
 'adapter_statuses': ['paper_strict_adapter'],
 'analysis_layers': ['regime_metrics',
  'window_metrics',
  'window_regime_metrics',
  'qlib_risk_metrics',
  'model_effect_deltas',
  'model_regime_window_diagnostics',
  'objective_scores',
  'promotion_scores',
  'model_comparison',
  'qlib_return_frame',
  'segment_metrics',
  'segment_effect_matrix',
  'model_review_briefing',
  'failure_modes'],
 'analysis_overview': {'baseline_model': 'hist_gradient_mse',
  'dashboard_note': '统一 regime/window 模型差异摘要，供 dashboard 总览和 schedule 通知读取。',
  'decision_hint_counts': {'block_or_rework': 84},
  'largest_relative_return_spread_segments': [{'baseline_model': 'hist_gradient_mse',
    'best_relative_return_model': 'hist_gradient_mse',
    'best_relative_return_vs_baseline_delta': 0.0,
    'best_stability_model': 'extra_trees_rank',
    'highest_drawdown_model': 'hist_gradient_mse',
    'highest_drawdown_pct': 5.588643182857744,
    'regime': 'downtrend',

In [5]:
notebook_summary = {
    "schema_version": "research.extra_trees_rank_native_adapter.v1",
    "model": "extra_trees_rank",
    "adapter_status": model_adapter_status("extra_trees_rank"),
    "model_adapter": model_adapter_name("extra_trees_rank"),
    "estimator_class": estimator.__class__.__name__,
    "target_kind": estimator.target_kind_,
    "rank_target_mean": estimator.rank_target_mean_,
    "rank_target_std": estimator.rank_target_std_,
    "feature_count": estimator.feature_count_,
    "prediction_count": int(len(predictions)),
    "prediction_has_na": bool(predictions["prediction"].isna().any()),
    "full_model_fidelity": fidelity,
    "matrix_model_fidelity": matrix_summary["model_fidelity"],
}

(EXPERIMENT_DIR / "notebook_summary.json").write_text(
    json.dumps(notebook_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
notebook_summary

{'schema_version': 'research.extra_trees_rank_native_adapter.v1',
 'model': 'extra_trees_rank',
 'adapter_status': 'paper_strict_adapter',
 'model_adapter': 'extra_trees_rank_forest_rank_v1',
 'estimator_class': 'ExtraTreesRankRegressor',
 'target_kind': 'forward_return_rank',
 'rank_target_mean': -2.3693784062122245e-18,
 'rank_target_std': 1.0,
 'feature_count': 6,
 'prediction_count': 26,
 'prediction_has_na': False,
 'full_model_fidelity': {'schema_version': 'model.fidelity.v1',
  'model_count': 32,
  'fidelity_counts': {'native_graph': 9,
   'native_library': 10,
   'native_sequence': 7,
   'native_torch': 6},
  'proxy_models': [],
  'native_models': ['alstm_mse',
   'autoformer_mse',
   'catboost_rank',
   'darnn_mse',
   'dlinear_mse',
   'extra_trees_rank',
   'fedformer_mse',
   'gat_mse',
   'gcn_mse',
   'gru_mse',
   'hawkes_gru_mse',
   'hist_gradient_mse',
   'hypergraph_mse',
   'lightgbm_mse',
   'logistic_clf',
   'lstm_mse',
   'market_graph_mse',
   'mlp_mixer_mse',


## 中文结论

- 做了什么：`extra_trees_rank` 已从直接 `ExtraTreesRegressor` proxy 路径升级为 `ExtraTreesRankRegressor` 原生 ranking adapter，adapter 名称固定为 `extra_trees_rank_forest_rank_v1`。
- 核心指标：本 notebook 断言训练后 rank target 均值接近 `0`、标准差大于 `0`、预测无缺失；全 QuantBench 模型 fidelity 为 `native_library=10`、`native_sequence=7`、`native_torch=6`、`native_graph=9`、`proxy=0`。
- regime 差异：本 notebook 只验证 adapter 和小矩阵 smoke；完整区间和 regime 差异以 `research/quantbench_all_model_run_ledger_v1/` 与 `research/quantbench_market_regime_analysis_v1/` 的刷新结果为准。
- 是否晋升：不直接晋升 production。该变更只解决模型实现 fidelity，生产晋升仍需要更长 walk-forward、成本滑点、压力窗口和 broker paper/sim 验证。
- 风险和下一步：ExtraTrees ranking 仍可能在短窗 stress/bull 分段弱于基准模型，下一步用刷新后的 ledger 和 regime analysis 检查它在不同区间的收益、回撤和稳定性差异。